PRELIMINARY VERSION WAITING FOR REAL  CONSUMPTION DATA


In [1]:
"""
Financial Distress Prediction Pipeline for an Electricity Utility
==================================================================

Illustrative implementation of the methodology surveyed in the accompanying
literature review: "Predicting Financial Distress in the Cameroonian
Electricity Utility Sector: Assessing the Incremental Predictive Value of
Consumption Forecast-Derived Risk Features Using Explainable Machine
Learning."

This script demonstrates, end-to-end, the reproducible Python toolchain
identified across the reviewed studies:
    1. Synthetic data generation (stand-in for real ENEO/SOCADEL-style
       regional-agency or customer-segment panel data).
    2. Consumption forecasting (proxy for load/demand forecasting models).
    3. Forecast-derived risk-feature engineering (residuals, volatility,
       trend/seasonal decomposition, forecast-vs-billed gap) -- following
       the logic of the Montenegrin electricity-debt SVR study and the
       non-technical-loss / theft-detection literature.
    4. A BASELINE distress classifier using only classical financial ratios
       (Altman-style liquidity / leverage / profitability variables).
    5. An AUGMENTED classifier adding the forecast-derived risk features.
    6. An incremental-predictive-value test (nested model comparison,
       AUC delta, bootstrap confidence interval) -- following the template
       used in the earnings-call-text, sustainability-report, and
       AI-adoption-signal "incremental value" studies reviewed above.
    7. SHAP-based explainability (global + local) for the augmented model.

PANEL DESIGN RATIONALE: Cameroon has a single national electricity utility,
so a firm-year panel (one row per year) would yield far too few observations
for ML. This script instead simulates a *sub-national* panel -- one row per
(operating region / agency / customer segment) per month -- which is the
realistic unit of analysis for an actual thesis, given that ARSEL/SOCADEL
are far more likely to release aggregated regional or segment-level series
than firm-level financials alone. See REAL_DATA_FIELD_MAPPING below, and the
companion "ARSEL/SOCADEL Data Request Package" document, for how each
synthetic field maps onto data that would actually need to be requested.

NOTE: All data below is SYNTHETIC and for demonstration purposes only. It
is not derived from real ENEO/SOCADEL financial statements or consumption
records. Replace the data-generation section with real panel data before
using this pipeline for actual research or decision-making.

Dependencies:
    pip install pandas numpy scikit-learn xgboost shap imbalanced-learn --break-system-packages
"""

# ---------------------------------------------------------------------------
# REAL_DATA_FIELD_MAPPING
# ---------------------------------------------------------------------------
# Maps each synthetic column produced below to (a) the real-world field it
# stands in for, and (b) its likely source, per the Data Specification
# Annex in the companion data-request document. Replace `simulate_utility_panel`
# and `label_financial_distress` with a loader that populates these same
# column names from real data, and the rest of the pipeline runs unchanged.
REAL_DATA_FIELD_MAPPING = {
    "consumption_billed": "Monthly billed kWh by region/segment -> SOCADEL commercial/billing systems",
    "ntl_rate": "Metered-vs-billed consumption gap -> SOCADEL technical/loss-management dept.",
    "liquidity_ratio": "Current assets / current liabilities, from audited financials -> SOCADEL finance dept.",
    "leverage_ratio": "Total debt / total assets, from audited financials -> SOCADEL finance dept.",
    "profitability": "EBIT / total assets, from audited financials -> SOCADEL finance dept.",
    "receivables_days": "Accounts-receivable turnover, from audited financials -> SOCADEL finance dept.",
    "public_sector_share": "Share of billing owed by government/parastatal accounts -> SOCADEL commercial dept. + MINFI",
    "arrears_public": "Public-sector billing minus public-sector collections, cumulative -> SOCADEL commercial dept.",
    "distress_score": "Graded Z''-style distress score / ARSEL MAR shortfall assessment -> ARSEL tariff-compensation decisions",
}

import numpy as np
import pandas as pd
from dataclasses import dataclass

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, classification_report
from sklearn.linear_model import LogisticRegression

try:
    import xgboost as xgb
    _HAS_XGB = True
except ImportError:
    # Falls back to scikit-learn's histogram gradient boosting classifier
    # if xgboost is not installed. Behaviourally similar (tree-based
    # gradient boosting) and fully compatible with SHAP's explainer used
    # below. In production, `pip install xgboost` is recommended, as
    # XGBoost/LightGBM/CatBoost are the classifiers most consistently
    # reported as top performers across the reviewed distress literature.
    from sklearn.ensemble import HistGradientBoostingClassifier
    _HAS_XGB = False

try:
    import shap
    _HAS_SHAP = True
except ImportError:
    _HAS_SHAP = False

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)


# ---------------------------------------------------------------------------
# 1. SYNTHETIC DATA GENERATION
# ---------------------------------------------------------------------------
@dataclass
class SimConfig:
    n_units: int = 300          # e.g. regional agencies / customer segments / subsidiaries
    n_months: int = 36          # 3 years of monthly panel data
    seed: int = RANDOM_STATE


def simulate_utility_panel(cfg: SimConfig) -> pd.DataFrame:
    """
    Simulates a monthly panel of operating units (e.g. Eneo/SOCADEL regional
    agencies) with:
      - actual electricity consumption (MWh billed)
      - classical financial ratios (liquidity, leverage, profitability)
      - a latent 'distress' propensity partly driven by consumption
        forecast errors and collection performance, in line with the
        reviewed literature linking arrears/NTL to utility financial risk.
    """
    n, T = cfg.n_units, cfg.n_months
    t = np.arange(T)

    records = []
    for unit_id in range(n):
        base_load = rng.uniform(500, 5000)          # MWh baseline
        seasonal_amp = rng.uniform(0.05, 0.25) * base_load
        trend = rng.uniform(-0.002, 0.004) * base_load
        noise_scale = rng.uniform(0.03, 0.12) * base_load

        # true (latent) consumption process
        seasonal = seasonal_amp * np.sin(2 * np.pi * t / 12)
        consumption_true = base_load + trend * t + seasonal
        consumption_true = np.clip(consumption_true, a_min=50, a_max=None)

        # Non-technical losses / collection friction shrink billed consumption
        ntl_rate = rng.uniform(0.02, 0.35)  # consistent with reported
                                             # Sub-Saharan African NTL ranges
        collection_shock = rng.normal(0, noise_scale, size=T)
        consumption_billed = consumption_true * (1 - ntl_rate) + collection_shock
        consumption_billed = np.clip(consumption_billed, a_min=10, a_max=None)

        # classical financial ratios (unit-level, slowly evolving)
        liquidity_ratio = rng.normal(1.1, 0.35, size=T).cumsum() / (t + 1) + 1.0
        leverage_ratio = rng.normal(0.55, 0.15, size=T).cumsum() / (t + 1) + 0.3
        profitability = rng.normal(0.04, 0.05, size=T).cumsum() / (t + 1)
        receivables_days = rng.normal(60, 20, size=T).cumsum() / (t + 1) + 30

        # public-sector arrears share (Cameroon-specific institutional signal)
        public_sector_share = rng.uniform(0.05, 0.4)
        arrears_public = rng.normal(0.02, 0.01, size=T).cumsum() * public_sector_share

        for m in range(T):
            records.append({
                "unit_id": unit_id,
                "month": m,
                "consumption_true": consumption_true[m],
                "consumption_billed": consumption_billed[m],
                "ntl_rate": ntl_rate,
                "liquidity_ratio": liquidity_ratio[m],
                "leverage_ratio": leverage_ratio[m],
                "profitability": profitability[m],
                "receivables_days": receivables_days[m],
                "public_sector_share": public_sector_share,
                "arrears_public": arrears_public[m],
            })

    df = pd.DataFrame.from_records(records)
    return df


def label_financial_distress(df: pd.DataFrame) -> pd.DataFrame:
    """
    Constructs a synthetic, unit-level distress indicator for the FINAL
    month of the panel, as a function of classical ratios PLUS consumption/
    collection frictions -- so that consumption-forecast-derived features
    carry genuine (simulated) incremental signal, mirroring the reviewed
    "incremental predictive value" literature design.

    Returns BOTH:
      - `distress_score`: a continuous, Z''-style graded risk score. With a
        single national utility and limited firm-years, a graded score
        (usable directly, or thresholded at multiple cut-points) is more
        robust for a real thesis than forcing a single binary "failure"
        event, which Cameroon has no clean historical record of.
      - `distress`: a binary label thresholded from that score, provided so
        the rest of the pipeline (AUC, SHAP, etc.) can run as a standard
        classification demo. In a real study, prefer reporting results
        across a few threshold choices (e.g. top 15%, 22%, 30% at risk)
        as a robustness check, rather than a single fixed cut-point.
    """
    last = df[df["month"] == df["month"].max()].copy()

    # latent risk score combining classical + consumption-side drivers
    z = (
        -1.8 * (last["liquidity_ratio"] - 1.0)
        + 2.2 * (last["leverage_ratio"] - 0.5)
        - 1.5 * last["profitability"]
        + 0.02 * (last["receivables_days"] - 60)
        + 1.6 * last["ntl_rate"]
        + 1.3 * last["arrears_public"]
        + rng.normal(0, 0.6, size=len(last))
    )
    prob = 1 / (1 + np.exp(-z))
    last["distress_score"] = prob  # continuous, 0-1 graded risk score
    last["distress"] = (prob > np.quantile(prob, 0.78)).astype(int)  # ~22% base rate
    return last[["unit_id", "distress_score", "distress"]]


# ---------------------------------------------------------------------------
# 2. CONSUMPTION FORECASTING (proxy for a load/demand forecasting model)
# ---------------------------------------------------------------------------
def forecast_consumption(unit_df: pd.DataFrame, horizon: int = 6) -> pd.DataFrame:
    """
    Simple, transparent seasonal-naive + linear-trend forecaster used here
    as a stand-in for a full production forecasting model (e.g. SARIMA,
    Prophet, or an ML regressor per the demand-forecasting literature).
    Produces monthly forecasts and forecast errors for the last `horizon`
    months of the panel, which are then engineered into risk features.
    """
    unit_df = unit_df.sort_values("month").reset_index(drop=True)
    y = unit_df["consumption_billed"].values
    train_end = len(y) - horizon

    # trend from training portion
    x_train = np.arange(train_end)
    coeffs = np.polyfit(x_train, y[:train_end], deg=1)
    trend_fn = np.poly1d(coeffs)

    # seasonal-naive component (average monthly deviation from trend, lag-12)
    detrended = y[:train_end] - trend_fn(x_train)
    seasonal_avg = pd.Series(detrended).groupby(x_train % 12).mean()

    forecasts = []
    for h in range(horizon):
        idx = train_end + h
        seasonal_component = seasonal_avg.get(idx % 12, 0.0)
        yhat = trend_fn(idx) + seasonal_component
        forecasts.append(yhat)

    actual = y[train_end:]
    forecast_arr = np.array(forecasts)
    residuals = actual - forecast_arr

    return pd.DataFrame({
        "unit_id": unit_df["unit_id"].iloc[0],
        "month": unit_df["month"].iloc[train_end:].values,
        "actual": actual,
        "forecast": forecast_arr,
        "residual": residuals,
    })


# ---------------------------------------------------------------------------
# 3. FORECAST-DERIVED RISK FEATURE ENGINEERING
# ---------------------------------------------------------------------------
def engineer_forecast_risk_features(panel: pd.DataFrame, horizon: int = 6) -> pd.DataFrame:
    """
    For each unit, fits the consumption forecaster and derives a
    unit-level risk-feature vector from the forecast errors:
        - mean absolute percentage residual (systematic under-collection)
        - residual volatility (instability of billing relative to forecast)
        - share of months with large negative residual (chronic shortfall)
        - trend in residuals (deteriorating vs improving collection gap)
    These mirror the "forecast-derived risk features" motivated by the
    Montenegrin electricity-debt and non-technical-loss literatures.
    """
    feats = []
    for unit_id, g in panel.groupby("unit_id"):
        fc = forecast_consumption(g, horizon=horizon)
        pct_resid = fc["residual"] / np.maximum(fc["forecast"], 1e-6)

        feats.append({
            "unit_id": unit_id,
            "forecast_bias_pct": pct_resid.mean(),
            "forecast_volatility": pct_resid.std(),
            "shortfall_month_share": (pct_resid < -0.10).mean(),
            "forecast_trend": np.polyfit(np.arange(len(pct_resid)), pct_resid, 1)[0],
        })
    return pd.DataFrame(feats)


# ---------------------------------------------------------------------------
# 4-6. BASELINE vs AUGMENTED MODELS + INCREMENTAL VALUE TEST
# ---------------------------------------------------------------------------
def build_feature_sets(panel: pd.DataFrame, labels: pd.DataFrame, horizon: int = 6):
    last = panel[panel["month"] == panel["month"].max()].copy()
    baseline_cols = ["liquidity_ratio", "leverage_ratio", "profitability",
                      "receivables_days", "public_sector_share"]
    baseline = last[["unit_id"] + baseline_cols].merge(labels, on="unit_id")

    forecast_feats = engineer_forecast_risk_features(panel, horizon=horizon)
    augmented = baseline.merge(forecast_feats, on="unit_id")
    return baseline, augmented, baseline_cols


def evaluate_incremental_value(baseline: pd.DataFrame, augmented: pd.DataFrame,
                                baseline_cols, n_bootstrap: int = 200):
    """
    Nested comparison: trains a baseline (financial-ratios-only) XGBoost
    classifier and an augmented (financial ratios + forecast-derived risk
    features) classifier, then compares out-of-sample AUC with a bootstrap
    confidence interval on the AUC delta -- the evaluation design used
    across the reviewed incremental-predictive-value studies.
    """
    forecast_cols = ["forecast_bias_pct", "forecast_volatility",
                      "shortfall_month_share", "forecast_trend"]

    X_base = baseline[baseline_cols].values
    X_aug = augmented[baseline_cols + forecast_cols].values
    y = augmented["distress"].values

    idx = np.arange(len(y))
    idx_train, idx_test = train_test_split(
        idx, test_size=0.3, stratify=y, random_state=RANDOM_STATE
    )

    def fit_predict(X):
        if _HAS_XGB:
            model = xgb.XGBClassifier(
                n_estimators=200, max_depth=3, learning_rate=0.05,
                subsample=0.8, colsample_bytree=0.8,
                eval_metric="logloss", random_state=RANDOM_STATE,
            )
        else:
            model = HistGradientBoostingClassifier(
                max_depth=3, learning_rate=0.05, max_iter=200,
                random_state=RANDOM_STATE,
            )
        model.fit(X[idx_train], y[idx_train])
        proba = model.predict_proba(X[idx_test])[:, 1]
        return model, proba

    model_base, proba_base = fit_predict(X_base)
    model_aug, proba_aug = fit_predict(X_aug)

    auc_base = roc_auc_score(y[idx_test], proba_base)
    auc_aug = roc_auc_score(y[idx_test], proba_aug)

    # bootstrap CI on the AUC delta
    deltas = []
    y_test = y[idx_test]
    for _ in range(n_bootstrap):
        boot_idx = rng.choice(len(y_test), size=len(y_test), replace=True)
        if len(np.unique(y_test[boot_idx])) < 2:
            continue
        d = (roc_auc_score(y_test[boot_idx], proba_aug[boot_idx])
             - roc_auc_score(y_test[boot_idx], proba_base[boot_idx]))
        deltas.append(d)
    ci_low, ci_high = np.percentile(deltas, [2.5, 97.5])

    print("=== Incremental Predictive Value: Baseline vs. Augmented ===")
    print(f"Baseline (financial ratios only)      AUC: {auc_base:.3f}")
    print(f"Augmented (+ consumption-forecast feats) AUC: {auc_aug:.3f}")
    print(f"AUC delta: {auc_aug - auc_base:+.3f}  "
          f"(95% bootstrap CI: [{ci_low:+.3f}, {ci_high:+.3f}])")

    return {
        "model_base": model_base, "model_aug": model_aug,
        "X_aug": X_aug, "idx_test": idx_test, "y_test": y_test,
        "feature_names": baseline_cols + forecast_cols,
        "auc_base": auc_base, "auc_aug": auc_aug,
    }


# ---------------------------------------------------------------------------
# 7. SHAP EXPLAINABILITY (global + local) FOR THE AUGMENTED MODEL
# ---------------------------------------------------------------------------
def explain_augmented_model(results: dict):
    model_aug = results["model_aug"]
    X_test = results["X_aug"][results["idx_test"]]
    y_test = results["y_test"]
    feature_names = results["feature_names"]

    if not _HAS_SHAP:
        # Fallback when the shap package is not installed in this environment:
        # scikit-learn permutation importance gives a model-agnostic, though
        # less granular (global-only, no local/instance-level) substitute.
        from sklearn.inspection import permutation_importance
        r = permutation_importance(
            model_aug, X_test, y_test, n_repeats=20,
            random_state=RANDOM_STATE, scoring="roc_auc",
        )
        ranking = sorted(zip(feature_names, r.importances_mean),
                          key=lambda x: -x[1])
        print("\n=== Permutation Importance (SHAP not installed -- fallback) ===")
        print("Install `shap` for full global + local, game-theoretic")
        print("explanations as described in Section 4 of the literature review.")
        for name, val in ranking:
            print(f"{name:24s} {val:.4f}")
        return None

    try:
        explainer = shap.Explainer(model_aug, feature_names=feature_names)
        shap_values = explainer(X_test)
    except Exception:
        # Robust fallback: permutation explainer works for any predict_proba model
        explainer = shap.Explainer(
            model_aug.predict_proba, X_test, feature_names=feature_names
        )
        shap_values = explainer(X_test)

    mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
    ranking = sorted(zip(feature_names, mean_abs_shap),
                      key=lambda x: -x[1])

    print("\n=== SHAP Global Feature Importance (Augmented Model) ===")
    for name, val in ranking:
        print(f"{name:24s} {val:.4f}")

    return shap_values


# ---------------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    cfg = SimConfig()
    panel = simulate_utility_panel(cfg)
    labels = label_financial_distress(panel)

    baseline_df, augmented_df, baseline_cols = build_feature_sets(panel, labels)
    results = evaluate_incremental_value(baseline_df, augmented_df, baseline_cols)
    explain_augmented_model(results)


=== Incremental Predictive Value: Baseline vs. Augmented ===
Baseline (financial ratios only)      AUC: 0.476
Augmented (+ consumption-forecast feats) AUC: 0.561
AUC delta: +0.084  (95% bootstrap CI: [-0.002, +0.175])

=== Permutation Importance (SHAP not installed -- fallback) ===
Install `shap` for full global + local, game-theoretic
explanations as described in Section 4 of the literature review.
liquidity_ratio          0.0359
forecast_trend           0.0310
forecast_volatility      0.0300
profitability            0.0134
leverage_ratio           0.0128
receivables_days         -0.0099
shortfall_month_share    -0.0104
forecast_bias_pct        -0.0152
public_sector_share      -0.0445
